In [53]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import os
import random
import copy

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(0)
random.seed(0)
print(f"Using device: {DEVICE}")

# Load BBC News Summary data
base_dir = "BBC News Summary"
articles_dir = os.path.join(base_dir, "News Articles")
summaries_dir = os.path.join(base_dir, "Summaries")

pairs = []
max_article_len = 20
max_summary_len = 5

categories = os.listdir(articles_dir)
for cat in categories:
    cat_article_dir = os.path.join(articles_dir, cat)
    cat_summary_dir = os.path.join(summaries_dir, cat)
    if not os.path.isdir(cat_article_dir):
        continue
    for fname in sorted(os.listdir(cat_article_dir)):
        if not fname.endswith(".txt"):
            continue
        article_path = os.path.join(cat_article_dir, fname)
        summary_path = os.path.join(cat_summary_dir, fname)
        if not os.path.exists(summary_path):
            continue
        with open(article_path, "r", encoding="latin-1") as f:
            lines = f.read().strip().split("\n")
        article = " ".join(lines[2:]) if len(lines) > 2 else lines[0]
        article_words = article.split()[:max_article_len]
        if len(article_words) < 5:
            continue
        with open(summary_path, "r", encoding="latin-1") as f:
            summary = f.read().strip()
        summary_words = summary.split()[:max_summary_len]
        if len(summary_words) < 3:
            continue
        pairs.append((" ".join(article_words).lower(), " ".join(summary_words).lower()))

print(f"Loaded {len(pairs)} article-summary pairs")

# Use a smaller subset for faster training
random.shuffle(pairs)
pairs = pairs[:100]
train_split = int(0.7 * len(pairs))
val_split = int(0.85 * len(pairs))
train_pairs = pairs[:train_split]
val_pairs = pairs[train_split:val_split]
test_pairs = pairs[val_split:]
print(f"Training pairs: {len(train_pairs)}, Validation pairs: {len(val_pairs)}, Test pairs: {len(test_pairs)}")

Using device: cpu
Loaded 2225 article-summary pairs
Training pairs: 70, Validation pairs: 15, Test pairs: 15


In [54]:
# Build vocabulary
SPECIAL = ["<PAD>", "<SOS>", "<EOS>", "<UNK>"]
words = set()
for a, b in pairs:
    words.update(a.split())
    words.update(b.split())
vocab = SPECIAL + sorted(words)
stoi = {w: i for i, w in enumerate(vocab)}
itos = {i: w for w, i in stoi.items()}
vocab_size = len(vocab)
print(f"Vocabulary size: {vocab_size}")

def encode(text, add_sos=False, add_eos=False):
    ids = []
    if add_sos:
        ids.append(stoi["<SOS>"])
    for w in text.split():
        ids.append(stoi.get(w, stoi["<UNK>"]))
    if add_eos:
        ids.append(stoi["<EOS>"])
    return torch.tensor(ids, dtype=torch.long)

train_data = [(encode(a), encode(b, True, True)) for a, b in train_pairs]
val_data = [(encode(a), encode(b, True, True)) for a, b in val_pairs]
test_data = [(encode(a), encode(b, True, True)) for a, b in test_pairs]

Vocabulary size: 1175


In [55]:
EMBEDDING_SIZE = 16
HIDDEN_SIZE = 32
VOCAB_SIZE = vocab_size
TARGET_LEN = 7
BATCH_SIZE = 1

In [56]:
shared_embedding = nn.Embedding(VOCAB_SIZE, EMBEDDING_SIZE)

In [57]:
class Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = shared_embedding
        self.rnn = nn.RNN(EMBEDDING_SIZE, HIDDEN_SIZE, batch_first=True)

    def forward(self, x):
        e = self.embedding(x)
        outputs, hidden = self.rnn(e)
        return outputs, hidden

In [58]:
class BahdanauAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.W_s = nn.Linear(HIDDEN_SIZE, HIDDEN_SIZE)
        self.W_h = nn.Linear(HIDDEN_SIZE, HIDDEN_SIZE)
        self.v = nn.Linear(HIDDEN_SIZE, 1)

    def forward(self, decoder_hidden, encoder_outputs, mask=None):
        query = decoder_hidden.unsqueeze(1)
        energy = torch.tanh(self.W_s(query) + self.W_h(encoder_outputs))  # (batch, seq_len, hidden)
        scores = self.v(energy).squeeze(-1)  # (batch, seq_len)

        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))

        attn_weights = F.softmax(scores, dim=1)  # (batch, seq_len)
        context = torch.bmm(attn_weights.unsqueeze(1), encoder_outputs)
        context = context.squeeze(1)  # (batch, hidden)
        return context, attn_weights

In [59]:
class Decoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = shared_embedding
        self.attention = BahdanauAttention()
        self.rnn = nn.RNN(EMBEDDING_SIZE+HIDDEN_SIZE, HIDDEN_SIZE, batch_first=True)
        self.fc_out = nn.Linear(HIDDEN_SIZE*2+EMBEDDING_SIZE, VOCAB_SIZE)


    def forward_step(self, input_token, hidden, encoder_outputs, mask=None):
        embedded = self.embedding(input_token.view(1, 1))
        context, attn_weights = self.attention(hidden.squeeze(0), encoder_outputs, mask)
        rnn_input = torch.cat((embedded, context.unsqueeze(1)), dim=2)
        output, hidden = self.rnn(rnn_input, hidden)
        output = output.squeeze(1)  # (batch, hidden)

        pred_input = torch.cat((output, context, embedded.squeeze(1)), dim=1)
        prediction = self.fc_out(pred_input)
        return prediction, hidden, attn_weights



    def forward(self, tgt, encoder_last_hidden, encoder_outputs, teacher_forcing_ratio=1.0):
        input_token = tgt[:, 0]
        hidden = encoder_last_hidden

        outputs = torch.zeros(BATCH_SIZE, TARGET_LEN, VOCAB_SIZE).to(DEVICE)

        for t in range(1, TARGET_LEN):
            # print(itos[input_token.item()])
            prediction, hidden, attn_weights = self.forward_step(input_token, hidden, encoder_outputs)
            outputs[:, t] = prediction

            use_teacher_forcing = random.random() < teacher_forcing_ratio
            input_token = tgt[:, t] if use_teacher_forcing else prediction.argmax(1)

        return outputs, attn_weights

In [60]:
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device

    def forward(self, src, tgt):
        encoder_outputs, encoder_hidden = self.encoder(src)
        outputs, attn_weights = self.decoder(tgt, encoder_hidden, encoder_outputs)
        return outputs, attn_weights


    @torch.no_grad()
    def greedy_decode(self, src, sos_idx, eos_idx, max_len=TARGET_LEN):
        self.eval()
        encoder_outputs, encoder_hidden = self.encoder(src)
        input_token = torch.tensor([sos_idx], device=self.device)

        tokens, attn_matrix = [], []
        for _ in range(max_len):
            prediction, hidden, attn_weights = self.decoder.forward_step(input_token, encoder_hidden, encoder_outputs)
            next_token = prediction.argmax(1)
            tokens.append(next_token.item())
            attn_matrix.append(attn_weights.squeeze(0).tolist())
            if next_token.item() == eos_idx:
                break
            input_token = next_token

        return tokens, attn_matrix

In [61]:
encoder = Encoder().to(DEVICE)
decoder = Decoder().to(DEVICE)
model = Seq2Seq(encoder, decoder, DEVICE).to(DEVICE)

optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss(ignore_index=stoi["<PAD>"])

In [62]:
def evaluate(data):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for src_batch, tgt_batch in data:
            src_batch = src_batch.unsqueeze(0).to(DEVICE)
            tgt_batch = tgt_batch.unsqueeze(0).to(DEVICE)
            outputs, _ = model(src_batch, tgt_batch)
            loss = criterion(outputs.permute(0, 2, 1), tgt_batch)
            total_loss += loss.item()
    return total_loss / len(data)

PATIENCE = 10
best_val_loss = float("inf")
epochs_no_improve = 0
best_model_state = None

for epoch in range(100):
    model.train()
    total_loss = 0
    for src_batch, tgt_batch in train_data:
        src_batch = src_batch.unsqueeze(0).to(DEVICE)
        tgt_batch = tgt_batch.unsqueeze(0).to(DEVICE)

        optimizer.zero_grad()
        outputs, _ = model(src_batch, tgt_batch)
        loss = criterion(outputs.permute(0, 2, 1), tgt_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    val_loss = evaluate(val_data)
    print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_data)}, Val Loss: {val_loss}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        epochs_no_improve = 0
        best_model_state = copy.deepcopy(model.state_dict())
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= PATIENCE:
            print(f"Early stopping at epoch {epoch+1} (no improvement for {PATIENCE} epochs)")
            break

if best_model_state is not None:
    model.load_state_dict(best_model_state)


Epoch 1, Loss: 6.8070401872907365, Val Loss: 6.170367940266927
Epoch 2, Loss: 5.343993180138725, Val Loss: 6.426571941375732
Epoch 3, Loss: 5.012696620396206, Val Loss: 6.502210966746012
Epoch 4, Loss: 4.778185680934361, Val Loss: 6.587913513183594
Epoch 5, Loss: 4.576383171762739, Val Loss: 6.701025867462159
Epoch 6, Loss: 4.403883729662214, Val Loss: 6.786697800954183
Epoch 7, Loss: 4.248926915441241, Val Loss: 6.856525516510009
Epoch 8, Loss: 4.103253793716431, Val Loss: 6.932943979899089
Epoch 9, Loss: 3.9611562422343662, Val Loss: 7.002173741658528
Epoch 10, Loss: 3.819564645630973, Val Loss: 7.055395476023356
Epoch 11, Loss: 3.677700400352478, Val Loss: 7.096619606018066
Early stopping at epoch 11 (no improvement for 10 epochs)


In [63]:
model.eval()
for src, tgt in test_data:
    src = src.unsqueeze(0).to(DEVICE)
    tgt = tgt.unsqueeze(0).to(DEVICE)
    predicted_tokens, attn_matrix = model.greedy_decode(src, stoi["<SOS>"], stoi["<EOS>"])
    predicted_summary = " ".join([itos[idx] for idx in predicted_tokens if idx not in (stoi["<SOS>"], stoi["<EOS>"], stoi["<PAD>"])])
    actual_summary = " ".join([itos[idx.item()] for idx in tgt[0] if idx.item() not in (stoi["<SOS>"], stoi["<EOS>"], stoi["<PAD>"])])
    actual_article = " ".join([itos[idx.item()] for idx in src[0] if idx.item() not in (stoi["<SOS>"], stoi["<EOS>"], stoi["<PAD>"])])
    print(f"Article: {actual_article}")
    print(f"Actual Summary: {actual_summary}")
    print(f"Predicted: {predicted_summary}")
    print("=" * 50)


Article: david beckham expressed his relief at real madrid's passage to the champions league knockout phase. after real's 3-0 win at
Actual Summary: david beckham expressed his relief
Predicted: 
Article: ireland maintained their six nations grand slam ambitions with an impressive victory over scotland at murrayfield. hugo southwell's try gave
Actual Summary: ireland got themselves on the
Predicted: 
Article: double olympic champion kelly holmes has been voted european athletics (eaa) woman athlete of 2004 in the governing body's annual
Actual Summary: double olympic champion kelly holmes
Predicted: 
Article: church ministers are trying to prevent rapper nelly performing in arkansas, saying they do not want his "vile and filthy
Actual Summary: "tear the tickets up," mr
Predicted: 
Article: motley crue guitarist mick mars is being sued by his ex-girlfriend for $10 million (â£5.4 million), claiming he broke a
Actual Summary: motley crue guitarist mick mars
Predicted: 
Article: film star 